# License plate Recognition ONLY Numbers VLM

Import Required Libraries

In [ ]:
import os 
import requests
import base64
import csv
import json
import pandas as pd
import Levenshtein
import re
import matplotlib.pyplot as plt

from difflib import SequenceMatcher
from PIL import Image

Configuration

In [ ]:
DATASET = r"your dataset path"
LMSTUDIO_URL = "your LMSTUDIO host link"
API_KEY = "your token"

headers = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json"
}

CER Function

In [ ]:
def cer_score(gt: str, pred: str) -> float:
    """
    Hitung Character Error Rate (CER) antara ground truth dan prediksi.
    
    CER = (S + D + I) / N
    - S = jumlah substitusi
    - D = jumlah penghapusan
    - I = jumlah penyisipan
    - N = panjang ground truth
    """
    distance = Levenshtein.distance(gt, pred)
    N = len(gt)
    return distance / N if N > 0 else 0.0


In [ ]:
def normalize_text(text: str) -> str:
    """Hilangkan karakter non-alfanumerik dan ubah ke huruf besar."""
    return re.sub(r"[^A-Z0-9]", "", text.upper())

def cer_score(gt: str, pred: str) -> float:
    gt_norm = normalize_text(gt)
    pred_norm = normalize_text(pred)
    distance = Levenshtein.distance(gt_norm, pred_norm)
    N = len(gt_norm)
    return distance / N if N > 0 else 0.0

loop all images on folder test

In [ ]:
results = []

for img_file in os.listdir(DATASET):
    if img_file.endswith(('.jpg', '.jpeg', '.png')):
        ground_truth = os.path.splitext(img_file)[0]

        with open(os.path.join(DATASET, img_file), "rb") as f:
            img_bytes = f.read()
        img_b64 = base64.b64encode(img_bytes).decode("utf-8")

        payload = {
            "model": "llava-phi-3-mini",
            "messages": [
                {
                    "role": "user",
                    "content": [
                        {"type": "text", "text": "OCR task: return ONLY the license plate characters (e.g., B1234XYZ). Do not include spaces, newlines, dates, or descriptions."},
                        {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{img_b64}"}}
                    ]
                }
            ]
        }

        response = requests.post(
            LMSTUDIO_URL,
            headers=headers,
            json=payload
        )

        resp_json = response.json()
        print("Status:", response.status_code)
        print("Response:", str(resp_json)[:200])

        prediction = resp_json["choices"][0]["message"]["content"]
        cer = cer_score(ground_truth, prediction)
        results.append([img_file, ground_truth, prediction, cer])

Put all results on CSV file

In [ ]:
df = pd.DataFrame(results, columns=["image", "ground_truth", "prediction", "CER_score"])
df.to_csv("results.csv", index=False)
df.head()

Analyze Result

In [ ]:
print("Mean CER:", df["CER_score"].mean())
print("Median CER:", df["CER_score"].median())

plt.figure(figsize=(10,5))
plt.scatter(range(len(df)), df["CER_score"], alpha=0.6)
plt.axhline(df["CER_score"].median(), color="red", linestyle="--", label="Median CER")
plt.title("Scatter Plot CER per Sampel")
plt.xlabel("Index Sampel")
plt.ylabel("CER")
plt.legend()
plt.show()
